# 自定义模型分级评估 

**注意：这节课的代码文件在同一个文件夹中。如果你想跟着操作并自行运行评估，请下载整个文件夹**


在这节课中，我们将学习如何使用 promptfoo 编写自定义模型分级评估。我们从一个简单的提示词目标开始：我们想写一个提示词，将冗长、复杂的技术维基百科文章转化为适合小学受众的简短摘要。

例如，给定整个[维基百科关于卷积神经网络的条目](https://en.wikipedia.org/wiki/Convolutional_neural_network)，我们希望得到如下简单的输出摘要：

> 卷积神经网络（CNN）是一种特殊类型的计算机程序，可以学习识别图像和模式。它们的工作方式有点像人脑，使用多层人工"神经元"来处理信息。
CNN 在识别图片中的物体或识别人脸等任务上表现出色。它们通过将图像分解成小块并寻找重要特征来实现这一点，就像拼拼图一样。
CNN 的特别之处在于它们可以通过观察大量示例来自己学习这些特征。这使它们能够越来越擅长识别事物，有时甚至可以达到人类水平的性能。
科学家和工程师将 CNN 用于各种有趣的应用，比如帮助自动驾驶汽车观察道路、发现新药物，甚至教计算机下棋和围棋等游戏。

为了评估提示词的有效性，我们将编写一个自定义模型分级断言，从三个方面评估生成的摘要：

* 简洁性（1-5）- 摘要是否尽可能简洁？
* 准确性（1-5）- 摘要是否完全基于原始文章准确？
* 语调（1-5）- 摘要是否适合没有任何技术背景的小学学生？

每个指标都会产生 1 到 5 之间的分数。我们将它们平均，目标是平均分至少达到 4.5/5。为此，我们需要定义一个自定义模型评分函数！

---
    


## 输入数据

我们的目标是编写一个提示词，将复杂的维基百科文章摘要成简短、易于理解的摘要。我们首先收集作为评估一部分的文章。

在这个文件夹中，我们提供了一个 `articles` 目录，其中包含八个不同的 txt 文件。每个文件包含一篇维基百科文章的文本内容。我们将使用这些文章作为评估的输入。查看一些文章文件，了解它们的长度和复杂程度。

这个数据集只包含八个测试用例，对于真实世界的评估来说远远不够。正如我们在本课程中多次提到的，我们强烈建议使用至少 100 个条目的评估数据集。

---

## 我们的提示词

查看 `prompts.py` 文件。它包含三个不同的提示词生成函数，我们将使用 promptfoo 对它们进行评估：

```py
def basic_summarize(article):
  return f"Summarize this article {article}"

def better_summarize(article):
  return f"""
  Summarize this article for a grade-school audience: {article}"""

def best_summarize(article):
  return f"""
  You are tasked with summarizing long wikipedia articles for a grade-school audience.
  Write a short summary, keeping it as concise as possible. 
  The summary is intended for a non-technical, grade-school audience. 
  This is the article: {article}"""
```
**需要注意的是，这些提示词总体上都是比较一般的提示词。我们故意保持提示词简短，没有遵循最佳实践（比如添加全面的示例），以尽量减少运行此评估集时使用的 token 数量。** 

---

## 更新配置文件

`promptfooconfig.yaml` 文件包含我们之前见过的大部分字段：


```yaml
description: 'Summarization Evaluation'

prompts:
  - prompts.py:basic_summarize
  - prompts.py:better_summarize
  - prompts.py:best_summarize

providers:
  - id: anthropic:messages:claude-3-5-sonnet-20240620
    label: "3.5 Sonnet"

tests:
  - vars:
      article: file://articles/article1.txt
  - vars:
      article: file://articles/article2.txt
  - vars:
      article: file://articles/article3.txt
  - vars:
      article: file://articles/article4.txt
  - vars:
      article: file://articles/article5.txt
  - vars:
      article: file://articles/article6.txt
  - vars:
      article: file://articles/article7.txt
  - vars:
      article: file://articles/article8.txt

defaultTest:
  assert:
    - type: python
      value: file://custom_llm_eval.py

```

我们告诉 promptfoo，要使用 `prompts.py` 中定义的三个提示词。接下来，我们配置 promptfoo 使用 Claude 3.5 Sonnet 作为 provider。

我们编写了一系列 `tests`，在每个测试中为 `article` 提供不同的值。这里唯一的新内容是，我们从文本文件加载值。文章非常长，在 YAML 文件中内联放置不太合理。例如，配置文件的这一部分：

```yaml
tests:
  - vars:
      article: file://articles/article1.txt
```

告诉 promptfoo，我们希望运行一个测试，其中 `article` 变量设置为 `article1.txt` 文件的文本内容。我们对所有八个文章文件重复此过程。

---

## 编写自定义模型评分函数

接下来，让我们关注 YAML 文件的最后一个字段：

```yaml
defaultTest:
  assert:
    - type: python
      value: file://custom_llm_eval.py
```

这个字段告诉 promptfoo，对于每个测试，我们都要运行 `custom_llm_eval.py` 文件中定义的特定 python 断言。我们之前在定义自定义代码分级断言时见过此语法。唯一的区别是，这次我们要编写一个使用另一个模型对模型输出进行评分的函数。

让我们看看 `custom_llm_eval.py` 文件的内容。它包含相当多的代码：

```py
import anthropic
import os
import json

def llm_eval(summary, article):
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    prompt = f"""Evaluate the following summary based on these criteria:
    1. Conciseness (1-5) - is the summary as concise as possible?
        - Conciseness of 1: The summary is unnecessarily long, including excessive details, repetitions, or irrelevant information. It fails to distill the key points effectively.
        - Conciseness of 3:  The summary captures most key points but could be more focused. It may include some unnecessary details or slightly over explain certain concepts.
        - Conciseness of 5: The summary effectively condenses the main ideas into a brief, focused text. It includes all essential information without any superfluous details or explanations.
    2. Accuracy (1-5) - is the summary completely accurate based on the initial article'?
        - Accuracy of 1: The summary contains significant errors, misrepresentations, or omissions that fundamentally alter the meaning or key points of the original article.
        - Accuracy of 3:  The summary captures some key points correctly but may have minor inaccuracies or omissions. The overall message is generally correct, but some details may be wrong.
        - Accuracy of 5: The summary faithfully represents the main gist of the original article without any errors or misinterpretations. All included information is correct and aligns with the source material.
    3. Tone (1-5) - is the summary appropriate for a grade school student with no technical training?
        - Tone of 1: The summary uses language or concepts that are too complex, technical, or mature for a grade school audience. It may contain jargon, advanced terminology, or themes that are not suitable for young readers.
        - Tone of 2:  The summary mostly uses language suitable for grade school students but occasionally includes terms or concepts that may be challenging. Some explanations might be needed for full comprehension.
        - Tone of 3: The summary consistently uses simple, clear language that is easily understandable by grade school students. It explains complex ideas in a way that is accessible and engaging for young readers.
    4. Explanation - a general description of the way the summary is evaluated

    <examples>
    <example>
    This summary:
    <summary>
    Artificial neural networks are computer systems inspired by how the human brain works. They are made up of interconnected "neurons" that process information. These networks can learn to do tasks by looking at lots of examples, similar to how humans learn. 

    Some key things about neural networks:
    - They can recognize patterns and make predictions
    - They improve with more data and practice
    - They're used for things like identifying objects in images, translating languages, and playing games

    Neural networks are a powerful tool in artificial intelligence and are behind many of the "smart" technologies we use today. While they can do amazing things, they still aren't as complex or capable as the human brain.
    <summary>
    Should receive a 5 for tone, a 5 for accuracy, and a 5 for conciseness
    </example>

    <example>
    This summary:
    <summary>
    Here is a summary of the key points from the article on artificial neural networks (ANNs):

    1. ANNs are computational models inspired by biological neural networks in animal brains. They consist of interconnected artificial neurons that process and transmit signals.

    2. Basic structure:
    - Input layer receives data
    - Hidden layers process information 
    - Output layer produces results
    - Neurons are connected by weighted edges

    3. Learning process:
    - ANNs learn by adjusting connection weights
    - Use techniques like backpropagation to minimize errors
    - Can perform supervised, unsupervised, and reinforcement learning

    4. Key developments:
    - Convolutional neural networks (CNNs) for image processing
    - Recurrent neural networks (RNNs) for sequential data
    - Deep learning with many hidden layers

    5. Applications:
    - Pattern recognition, classification, regression
    - Computer vision, speech recognition, natural language processing
    - Game playing, robotics, financial modeling

    6. Advantages:
    - Can model complex non-linear relationships
    - Ability to learn and generalize from data
    - Adaptable to many different types of problems

    7. Challenges:
    - Require large amounts of training data
    - Can be computationally intensive
    - "Black box" nature can make interpretability difficult

    8. Recent advances:
    - Improved hardware (GPUs) enabling deeper networks
    - New architectures like transformers for language tasks
    - Progress in areas like generative AI

    The article provides a comprehensive overview of ANN concepts, history, types, applications, and ongoing research areas in this field of artificial intelligence and machine learning.
    </summary>
    Should receive a 1 for tone, a 5 for accuracy, and a 3 for conciseness
    </example>
    </examples>

    Provide a score for each criterion in JSON format. Here is the format you should follow always:

    <json>
    {{
    "conciseness": <number>,
    "accuracy": <number>,
    "tone": <number>,
    "explanation": <string>,
    }}
    </json>


    Original Text: <original_article>{article}</original_article>
    
    Summary to Evaluate: <summary>{summary}</summary>
    """
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=1000,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": "<json>" 
            }
        ],
        stop_sequences=["</json>"]
    )
    
    evaluation = json.loads(response.content[0].text)
    # Filter out non-numeric values and calculate the average
    numeric_values = [value for key, value in evaluation.items() if isinstance(value, (int, float))]
    avg_score = sum(numeric_values) / len(numeric_values)
    return avg_score, response.content[0].text

def get_assert(output: str, context, threshold=4.5):
    article = context['vars']['article']
    score, evaluation = llm_eval(output, article )
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }

```

### `get_assert()`

这里有很多内容需要讨论，但让我们从文件底部的函数开始：`get_assert`

```py
def get_assert(output: str, context, threshold=4.5):
    article = context['vars']['article']
    score, evaluation = llm_eval(output, article )
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }
```

回顾我们之前的课程，promptfoo 会自动在断言文件中查找名为 `get_assert` 的函数。它将向该函数传递以下两个参数：

- 给定模型响应的 `output`
- 包含生成输出的变量和提示词的 `context` 字典

promptfoo 期望我们的函数返回以下之一：
- 布尔值（通过/失败）
- 浮点数（分数）
- GradingResult 字典

我们选择返回 GradingResult 字典，它必须包含以下属性：

- `pass`：布尔值
- `score`：浮点数
- `reason`：字符串解释

下面是该函数的注释版本，解释了发生的事情：

```py
def get_assert(output: str, context, threshold=4.5):
    # 从 context 中获取特定的文章
    article = context['vars']['article']
    # 将模型输出和文章传递给一个名为 llm_eval 的函数
    score, evaluation = llm_eval(output, article ) # 捕获它返回的分数和评估解释
    # 返回一个字典，指示输出是否通过测试、其分数以及分数背后的解释
    return {
        "pass": score >= threshold,
        "score": score,
        "reason": evaluation
    }
```

### `llm_eval()`
接下来，让我们仔细看看实际进行评分的 `llm_eval` 函数。该函数执行以下操作：

1. 定义一个非常长的评分标准提示词，解释如何对摘要进行评分
2. 通过向 Anthropic API 发送请求来运行评分提示词
3. 解析响应并计算平均分数
4. 返回平均分数和模型的完整文本响应

以下是完整的代码：

```py
def llm_eval(summary, article):
    """
    使用 LLM（Claude）评估摘要。
    
    参数：
    summary (str): 要评估的摘要。
    article (str): 被摘要的原始文本。
    
    返回：
    bool: 如果平均分数高于阈值则返回 True，否则返回 False。
    """
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

    prompt = f"""Evaluate the following summary based on these criteria:
    1. Conciseness (1-5) - is the summary as concise as possible?
        - Conciseness of 1: The summary is unnecessarily long, including excessive details, repetitions, or irrelevant information. It fails to distill the key points effectively.
        - Conciseness of 3:  The summary captures most key points but could be more focused. It may include some unnecessary details or slightly overexplain certain concepts.
        - Conciseness of 5: The summary effectively condenses the main ideas into a brief, focused text. It includes all essential information without any superfluous details or explanations.
    2. Accuracy (1-5) - is the summary completely accurate based on the initial article'?
        - Accuracy of 1: The summary contains significant errors, misrepresentations, or omissions that fundamentally alter the meaning or key points of the original article.
        - Accuracy of 3:  The summary captures some key points correctly but may have minor inaccuracies or omissions. The overall message is generally correct, but some details may be wrong.
        - Accuracy of 5: The summary faithfully represents the main gist of the original article without any errors or misinterpretations. All included information is correct and aligns with the source material.
    4. Tone (1-5) - is the summary appropriate for a grade school student with no technical training?
        - Tone of 1: The summary uses language or concepts that are too complex, technical, or mature for a grade school audience. It may contain jargon, advanced terminology, or themes that are not suitable for young readers.
        - Tone of 2:  The summary mostly uses language suitable for grade school students but occasionally includes terms or concepts that may be challenging. Some explanations might be needed for full comprehension.
        - Tone of 3: The summary consistently uses simple, clear language that is easily understandable by grade school students. It explains complex ideas in a way that is accessible and engaging for young readers.
    5. Explanation - a general description of the way the summary is evaluated

    <examples>
    <example>
    This summary:
    <summary>
    Artificial neural networks are computer systems inspired by how the human brain works. They are made up of interconnected "neurons" that process information. These networks can learn to do tasks by looking at lots of examples, similar to how humans learn. 

    Some key things about neural networks:
    - They can recognize patterns and make predictions
    - They improve with more data and practice
    - They're used for things like identifying objects in images, translating languages, and playing games

    Neural networks are a powerful tool in artificial intelligence and are behind many of the "smart" technologies we use today. While they can do amazing things, they still aren't as complex or capable as the human brain.
    <summary>
    Should receive a 5 for tone, a 5 for accuracy, and a 5 for conciseness
    </example>

    <example>
    This summary:
    <summary>
    Here is a summary of the key points from the article on artificial neural networks (ANNs):

    1. ANNs are computational models inspired by biological neural networks in animal brains. They consist of interconnected artificial neurons that process and transmit signals.

    2. Basic structure:
    - Input layer receives data
    - Hidden layers process information 
    - Output layer produces results
    - Neurons are connected by weighted edges

    3. Learning process:
    - ANNs learn by adjusting connection weights
    - Use techniques like backpropagation to minimize errors
    - Can perform supervised, unsupervised, and reinforcement learning

    4. Key developments:
    - Convolutional neural networks (CNNs) for image processing
    - Recurrent neural networks (RNNs) for sequential data
    - Deep learning with many hidden layers

    5. Applications:
    - Pattern recognition, classification, regression
    - Computer vision, speech recognition, natural language processing
    - Game playing, robotics, financial modeling

    6. Advantages:
    - Can model complex non-linear relationships
    - Ability to learn and generalize from data
    - Adaptable to many different types of problems

    7. Challenges:
    - Require large amounts of training data
    - Can be computationally intensive
    - "Black box" nature can make interpretability difficult

    8. Recent advances:
    - Improved hardware (GPUs) enabling deeper networks
    - New architectures like transformers for language tasks
    - Progress in areas like generative AI

    The article provides a comprehensive overview of ANN concepts, history, types, applications, and ongoing research areas in this field of artificial intelligence and machine learning.
    </summary>
    Should receive a 1 for tone, a 5 for accuracy, and a 3 for conciseness
    </example>
    </examples>

    Provide a score for each criterion in JSON format. Here is the format you should follow always:

    <json>
    {{
    "conciseness": <number>,
    "accuracy": <number>,
    "tone": <number>,
    "explanation": <string>,
    }}
    </json>


    Original Text: <original_article>{article}</original_article>
    
    Summary to Evaluate: <summary>{summary}</summary>
    """
    
    response = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=1000,
        temperature=0,
        messages=[
            {
                "role": "user",
                "content": prompt
            },
            {
                "role": "assistant",
                "content": "<json>" 
            }
        ],
        stop_sequences=["</json>"]
    )
    
    evaluation = json.loads(response.content[0].text)
    # Filter out non-numeric values and calculate the average
    numeric_values = [value for key, value in evaluation.items() if isinstance(value, (int, float))]
    avg_score = sum(numeric_values) / len(numeric_values)
    # Return the average score and the overall model response
    return avg_score, response.content[0].text
```

---

## 运行评估

我们使用之前见过的相同命令来运行评估：

```bash
npx promptfoo@latest eval
```
这个过程可能需要一些时间才能完成，因为我们要向模型发送初始请求来生成文章摘要，然后还要发送额外请求来对这些摘要进行评分！

这是我们获得的评估结果截图：



让我们启动 Web 视图以便更好地了解结果：

```bash
npx promptfoo@latest view
```
这是 Web 仪表板的截图：



我们可以点击每个单元格中的放大镜来查看测试结果的更多信息：



我们可以看到，这个特定输出没有通过我们的自定义 llm-eval 函数评估，因为它的语调分数非常低。

此外，结果的顶部行显示了每个提示词的评分摘要：



毫不奇怪，我们的 `best_summary` 提示词表现最好！

仪表板的顶部还显示了一些图表来帮助可视化分数：



在上面的截图中：

* 红色是我们的 `basic_summarize` 提示词
* 蓝色是我们的 `better_summarize` 提示词
* 绿色是我们的 `best_summarize` 提示词

图表显示，`best_summarize` 提示词不仅从未未通过我们的测试，而且在所有输入上都超过了其他提示词。